# UTS_SGS_InnoLab_C1 — Sub 5 (`ibm-granite/granite-3.3-2b-instruct` LoRA, Full 14,000 Rows, Single Model)

- **Model:** `ibm-granite/granite-3.3-2b-instruct` (single approved pretrained backbone, LoRA `r=16, alpha=32, dropout=0.05` + zero-init linear classification head on last token)
- **Training Data:** All 14,000 labelled rows (`train.csv`), raw text, head 75% + tail 25% truncation
- **Schedule (Kaggle Fast Lite + Time-Guarded):** 2.0 epochs (875 steps at effective batch size 32), cosine schedule, `lr=1e-4`, `weight_decay=0.01`, sqrt-inverse class weights `[1.00, 1.41, 2.98]`, `seed=42`
- **Team Rule:** Single model only — **no ensembles**.


In [ ]:
# Cell 1 — Remove incompatible torchao (if present) so peft imports cleanly
!pip uninstall -y -q torchao


In [ ]:
"""Cell 2 for Kaggle Notebook — Sub 5 Granite-3.3-2B LoRA Full-14k Preparation.

Auto-discovers train.csv, test.csv, folds.csv, sub3_submission.csv, and
ibm-granite/granite-3.3-2b-instruct from /kaggle/input/**, /content/**, or HuggingFace.
Automatically selects bf16 (on L4/A100 sm_80+) or safe fp16 with fp32 master LoRA weights
(on Kaggle T4 sm_75) and sets MAX_LEN / batch size so training finishes before 17:45 UTC+7.
"""
import gc
import glob
import hashlib
import os
import time

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
# Pin to single GPU so HF Trainer does not wrap PEFT + gradient_checkpointing in DataParallel
if "CUDA_VISIBLE_DEVICES" not in os.environ:
    os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import numpy as np
import pandas as pd
import peft
import torch
import torch.nn as nn
import transformers
from peft import LoraConfig, get_peft_model
from sklearn.metrics import f1_score
from transformers import (
    AutoModel,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)
from transformers.modeling_outputs import SequenceClassifierOutput

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
transformers.set_seed(42)

assert torch.cuda.is_available(), "No CUDA GPU detected! Enable GPU accelerator in Kaggle Notebook settings."
gpu_name = torch.cuda.get_device_name(0)
cc_major, cc_minor = torch.cuda.get_device_capability(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
USE_NATIVE_BF16 = cc_major >= 8  # Ampere/Ada (A100, L4) -> native bf16; Turing (T4) -> fp16 Tensor Cores
AMP_DTYPE = torch.bfloat16 if USE_NATIVE_BF16 else torch.float16

print(
    f"transformers {transformers.__version__} | peft {peft.__version__} | "
    f"GPU: {gpu_name} (sm_{cc_major}{cc_minor}, {vram_gb:.1f} GB) | "
    f"precision: {'bf16' if USE_NATIVE_BF16 else 'fp16'}"
)

LABELS = ["BENIGN", "JAILBREAK", "PROMPT_INJECTION"]
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for l, i in label2id.items()}
FOLD0_SHA = "0d043c7efa1612b707d4a52e4b73a0aa311ee37fb3f9cadf213ec47269a57fc4"


def find_file(filename, required=True):
    search_patterns = [
        f"/kaggle/input/**/{filename}",
        f"/kaggle/working/**/{filename}",
        f"/content/**/{filename}",
        f"data/{filename}",
        filename,
    ]
    for pattern in search_patterns:
        hits = sorted(glob.glob(pattern, recursive=True))
        if hits:
            return hits[0]
    if required:
        raise FileNotFoundError(
            f"Could not find {filename}. Attach the dataset containing {filename} under /kaggle/input/."
        )
    return None


def find_granite_model():
    for pattern in ["/kaggle/input/**/config.json", "/content/**/config.json"]:
        for cfg in sorted(glob.glob(pattern, recursive=True)):
            if "granite" in cfg.lower():
                return os.path.dirname(cfg)
    return "ibm-granite/granite-3.3-2b-instruct"


TRAIN_PATH = find_file("train.csv", required=True)
TEST_PATH = find_file("test.csv", required=True)
FOLDS_PATH = find_file("folds.csv", required=True)
SUB3_CSV = find_file("sub3_submission.csv", required=False)
MODEL_DIR = find_granite_model()

# On 16 GB T4, MAX_LEN=768 truncates only 1.66% of rows (head 75% + tail 25%) and runs ~35% faster.
# On >=24 GB L4/A100, MAX_LEN=1024 truncates 0.81% of rows.
MAX_LEN = 1024 if vram_gb >= 22.0 else 768
WORK_ROOT = "/kaggle/working" if os.path.exists("/kaggle/working") else "/content"
OUT = os.path.join(WORK_ROOT, "sub5_granite_full")
os.makedirs(OUT, exist_ok=True)

print("TRAIN_PATH :", TRAIN_PATH)
print("TEST_PATH  :", TEST_PATH)
print("FOLDS_PATH :", FOLDS_PATH)
print("SUB3_CSV   :", SUB3_CSV)
print("MODEL_DIR  :", MODEL_DIR)
print("MAX_LEN    :", MAX_LEN)
print("OUT        :", OUT)

read = lambda p: pd.read_csv(p, dtype=str, keep_default_na=False, encoding="utf-8")
train, test, folds = read(TRAIN_PATH), read(TEST_PATH), read(FOLDS_PATH)
folds["fold"] = folds["fold"].astype(int)
train = train.merge(folds, on="id", how="left", validate="one_to_one").reset_index(drop=True)
assert train["fold"].notna().all()

fold0 = sorted(train.loc[train.fold == 0, "id"])
assert (
    len(fold0) == 2800
    and hashlib.sha256(",".join(fold0).encode()).hexdigest() == FOLD0_SHA
), "fold-0 hash mismatch"

tr_mask = (train.fold != 0).values
va_mask = (train.fold == 0).values
tr_df = train[tr_mask].reset_index(drop=True)
va_df = train[va_mask].reset_index(drop=True)
print("rows (folds 1-4 / fold 0 / test / total train):", len(tr_df), len(va_df), len(test), len(train))

tok = AutoTokenizer.from_pretrained(MODEL_DIR)


def encode(texts, max_len=MAX_LEN, head_frac=0.75):
    enc = tok(list(texts), add_special_tokens=True, truncation=False)
    out, n_trunc = [], 0
    for ids in enc["input_ids"]:
        if len(ids) > max_len:
            h = int(max_len * head_frac)
            ids = ids[:h] + ids[-(max_len - h) :]
            n_trunc += 1
        out.append(ids)
    print(f"truncated {n_trunc}/{len(out)} ({n_trunc / len(out):.2%}) at max_len={max_len}")
    return out, n_trunc / len(out)


all_train_ids, TRUNC_RATE = encode(train.text)
tr_ids = [all_train_ids[i] for i in range(len(train)) if tr_mask[i]]
va_ids = [all_train_ids[i] for i in range(len(train)) if va_mask[i]]
te_ids, te_trunc = encode(test.text)


class DS(torch.utils.data.Dataset):
    def __init__(self, ids, labels=None):
        self.ids, self.labels = ids, labels

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        d = {"input_ids": self.ids[i], "attention_mask": [1] * len(self.ids[i])}
        if self.labels is not None:
            d["labels"] = int(self.labels[i])
        return d


tr_y = tr_df.label.map(label2id).tolist()
va_y = va_df.label.map(label2id).tolist()
tr_ds, va_ds, te_ds = DS(tr_ids, tr_y), DS(va_ids, va_y), DS(te_ids)
counts = tr_df.label.value_counts().reindex(LABELS).values
CLASS_W = torch.tensor(np.sqrt(counts.max() / counts), dtype=torch.float)
print("class weights", CLASS_W.tolist())


class GraniteClassifier(nn.Module):
    def __init__(self, path, num_labels=3):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(
            path, dtype=AMP_DTYPE, attn_implementation="sdpa"
        )
        self.score = nn.Linear(self.backbone.config.hidden_size, num_labels, bias=False)
        nn.init.zeros_(self.score.weight)

    def forward(self, input_ids, attention_mask, labels=None):
        with torch.amp.autocast("cuda", dtype=AMP_DTYPE):
            h = self.backbone(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        last = attention_mask.shape[1] - 1 - attention_mask.flip(1).argmax(1)
        pooled = h[torch.arange(h.size(0), device=h.device), last].float()
        pooled = torch.nan_to_num(pooled, nan=0.0, posinf=1e4, neginf=-1e4)
        logits = self.score(pooled)
        return SequenceClassifierOutput(logits=logits)


def build_model():
    model = GraniteClassifier(MODEL_DIR)
    model.backbone.config.use_cache = False
    model.backbone.gradient_checkpointing_enable(
        gradient_checkpointing_kwargs={"use_reentrant": False}
    )
    model.backbone = get_peft_model(
        model.backbone,
        LoraConfig(
            r=16,
            lora_alpha=32,
            lora_dropout=0.05,
            target_modules=[
                "q_proj",
                "k_proj",
                "v_proj",
                "o_proj",
                "gate_proj",
                "up_proj",
                "down_proj",
            ],
        ),
    )
    # Keep trainable LoRA + head weights in fp32 so GradScaler (fp16 mode on T4) never fails
    for p in model.parameters():
        if p.requires_grad:
            p.data = p.data.float()
    model.cuda()
    return model


class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = nn.functional.cross_entropy(
            outputs.logits.float(), labels, weight=CLASS_W.to(outputs.logits.device)
        )
        return (loss, outputs) if return_outputs else loss


def compute_metrics(p):
    logits = p.predictions[0] if isinstance(p.predictions, tuple) else p.predictions
    return {"macro_f1": f1_score(p.label_ids, np.argmax(logits, axis=-1), average="macro")}


t0 = time.time()
model = build_model()
print(
    f"model ready in {time.time() - t0:.0f}s | trainable "
    f"{sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6:.1f}M"
)
print("PREPARE_OK")


In [ ]:
"""Cell 3+5+6+7 for Kaggle Notebook — Sub 5 Granite-3.3-2B LoRA (Fast Lite + Time-Guarded).

Designed to finish before 17:45 UTC+7 on Kaggle:
1. Uses Sub 5 Lite schedule by default (MODE="full", all 14,000 rows, 2.0 epochs = 875 steps,
   cosine schedule, wd=0.01) so no second shadow GPU run is needed.
2. Evaluates test logits using the ordered inference loop (never trainer.predict) at
   epochs [1.0, 1.5, 2.0] and IMMEDIATELY writes a validated /kaggle/working/submission.csv
   at each snapshot so a submission-ready CSV is always on disk.
3. Enforces a wall-clock safety guard (17:25 UTC+7 / 10:25 UTC, or MAX_TRAIN_MINUTES=38):
   if training approaches the cutoff after >= 1.0 epoch, it stops cleanly, runs final ordered
   inference on test.csv, and writes /kaggle/working/submission.csv immediately.
"""
import datetime as dt
import json
import os
import time

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import f1_score
from transformers import DataCollatorWithPadding, TrainerCallback, TrainingArguments

MODE = "full"  # Train single model on all 14,000 labelled rows (no ensemble)
EPOCHS = 2.0   # 875 optimizer steps at effective batch 32 (~25-32 min)
SCHED = "cosine"
WD = 0.01

# Auto-size micro-batch for 16 GB T4 (8 x 4 = 32) vs >=24 GB L4/A100 (16 x 2 = 32)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
BS, GA = (16, 2) if vram_gb >= 22.0 else (8, 4)
INF_BS = 32 if vram_gb >= 22.0 else 16

# Snapshot only at 1.0, 1.5, 2.0 epochs (saves ~7 min vs 6 snapshots while keeping safety checkpoints)
GRID_E = [1.0, 1.5, 2.0]

# Hard wall-clock cutoff: 2026-10-22 10:25 UTC (17:25 UTC+7) or 38 minutes max runtime
HARD_CUTOFF_UTC = dt.datetime(2026, 10, 22, 10, 25, tzinfo=dt.timezone.utc)
MAX_TRAIN_MINUTES = 38.0

WORK_ROOT = "/kaggle/working" if os.path.exists("/kaggle/working") else "/content"
OUT5 = os.path.join(WORK_ROOT, f"sub5_granite_{MODE}")
os.makedirs(OUT5, exist_ok=True)
TOP_SUBMISSION_CSV = os.path.join(WORK_ROOT, "submission.csv")

fit_df = train.reset_index(drop=True) if MODE == "full" else tr_df
fit_ids = all_train_ids if MODE == "full" else tr_ids

counts = fit_df.label.value_counts().reindex(LABELS).values
CLASS_W = torch.tensor(np.sqrt(counts.max() / counts), dtype=torch.float)
fit_ds = DS(fit_ids, fit_df.label.map(label2id).tolist())
print(f"MODE={MODE} | rows={len(fit_df)} | BS={BS}x{GA} | class_weights={CLASS_W.tolist()}")

sub3_lab = None
if SUB3_CSV and os.path.exists(SUB3_CSV):
    s3 = pd.read_csv(SUB3_CSV, dtype=str)
    if s3.id.tolist() == test.id.tolist():
        sub3_lab = s3.label.map(label2id).values
        print(f"Loaded Sub 3 reference ({SUB3_CSV}) for agreement check.")

collate = DataCollatorWithPadding(tok)


def ordered_logits(m, ids, bs=INF_BS):
    """Length-sorted batches, results written back to the ORIGINAL row order."""
    was_training, order = m.training, np.argsort([len(x) for x in ids])
    out = np.zeros((len(ids), 3), dtype=np.float32)
    m.eval()
    with torch.no_grad():
        for s in range(0, len(ids), bs):
            idx = order[s : s + bs]
            b = collate([{"input_ids": ids[i], "attention_mask": [1] * len(ids[i])} for i in idx])
            logits = m(
                input_ids=b["input_ids"].cuda(),
                attention_mask=b["attention_mask"].cuda(),
            ).logits.float()
            out[idx] = torch.nan_to_num(logits, nan=0.0).cpu().numpy()
    m.train(was_training)
    return out


def write_validated_submission(te_logits, tag_epoch):
    pred = te_logits.argmax(1)
    sub = pd.DataFrame({"id": test.id, "label": [LABELS[i] for i in pred]})
    shares = np.bincount(pred, minlength=3) / len(pred)
    shares_ok = bool(np.abs(shares - np.array([0.62, 0.31, 0.07])).max() <= 0.03)
    agree_s3 = float((pred == sub3_lab).mean()) if sub3_lab is not None else None

    assert list(sub.columns) == ["id", "label"] and len(sub) == 6000 and sub.id.is_unique
    assert sub.id.tolist() == test.id.tolist() and sub.label.isin(LABELS).all()

    snap_csv = os.path.join(OUT5, f"submission_e{tag_epoch}.csv")
    sub.to_csv(snap_csv, index=False)
    # Always keep the latest valid submission at /kaggle/working/submission.csv
    sub.to_csv(TOP_SUBMISSION_CSV, index=False)
    sub.to_csv(os.path.join(OUT5, "submission.csv"), index=False)

    report = {
        "E_star": tag_epoch,
        "bias": [0.0, 0.0, 0.0],
        "labels_changed_by_bias": 0,
        "test_shares": shares.round(3).tolist(),
        "shares_ok": shares_ok,
        "agree_sub3": round(agree_s3, 4) if agree_s3 is not None else None,
        "submission_path": TOP_SUBMISSION_CSV,
    }
    json.dump(report, open(os.path.join(OUT5, "final_report.json"), "w"), indent=2)
    return report


class FastKaggleSnapshot(TrainerCallback):
    def on_train_begin(self, args, state, control, **kw):
        self.at = {round(state.max_steps * e / EPOCHS): e for e in GRID_E}
        self.recs = []
        self.t0 = time.time()
        self.emergency_saved = False
        print("max_steps:", state.max_steps, "| scheduled snapshots:", self.at)

    def _run_snapshot(self, e, step, model):
        mins = round((time.time() - self.t0) / 60, 1)
        te = ordered_logits(model, te_ids)
        np.save(os.path.join(OUT5, f"test_logits_e{e}.npy"), te)
        rep = write_validated_submission(te, e)
        rec = {
            "epoch": e,
            "step": step,
            "minutes": mins,
            "test_shares": rep["test_shares"],
            "shares_ok": rep["shares_ok"],
            "agree_sub3": rep["agree_sub3"],
        }
        self.recs.append(rec)
        json.dump(self.recs, open(os.path.join(OUT5, "snapshots.json"), "w"), indent=2)
        print(f"SNAPSHOT_SAVED -> {TOP_SUBMISSION_CSV} :", rec)

    def on_step_end(self, args, state, control, model=None, **kw):
        e = self.at.get(state.global_step)
        if e is not None:
            self._run_snapshot(e, state.global_step, model)
            if e == EPOCHS:
                d = os.path.join(OUT5, f"adapter_e{e}")
                model.backbone.save_pretrained(d)
                torch.save(model.score.state_dict(), os.path.join(d, "score.pt"))
            return

        # Time-cutoff guard: if past 1.0 epoch and approaching 17:25 UTC+7 or MAX_TRAIN_MINUTES
        cur_ep = round(EPOCHS * state.global_step / max(state.max_steps, 1), 2)
        elapsed_min = (time.time() - self.t0) / 60.0
        now_utc = dt.datetime.now(dt.timezone.utc)
        if (
            not self.emergency_saved
            and cur_ep >= 1.0
            and (elapsed_min >= MAX_TRAIN_MINUTES or now_utc >= HARD_CUTOFF_UTC)
        ):
            self.emergency_saved = True
            print(
                f"TIME_CUTOFF_GUARD triggered at step {state.global_step} "
                f"(epoch {cur_ep}, {elapsed_min:.1f} min). Saving final test predictions now..."
            )
            self._run_snapshot(cur_ep, state.global_step, model)
            d = os.path.join(OUT5, f"adapter_e{cur_ep}")
            model.backbone.save_pretrained(d)
            torch.save(model.score.state_dict(), os.path.join(d, "score.pt"))
            control.should_training_stop = True


args = TrainingArguments(
    output_dir=os.path.join(WORK_ROOT, f"sub5_{MODE}_tmp"),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BS,
    gradient_accumulation_steps=GA,
    learning_rate=1e-4,
    lr_scheduler_type=SCHED,
    warmup_ratio=0.05,
    weight_decay=WD,
    max_grad_norm=1.0,
    bf16=USE_NATIVE_BF16,
    fp16=not USE_NATIVE_BF16,
    group_by_length=True,
    eval_strategy="no",
    save_strategy="no",
    report_to="none",
    logging_steps=25,
    label_names=["labels"],
    dataloader_num_workers=2,
    seed=42,
)

trainer = WeightedTrainer(
    model=model,
    args=args,
    train_dataset=fit_ds,
    data_collator=collate,
    processing_class=tok,
    callbacks=[FastKaggleSnapshot()],
)

t0 = time.time()
trainer.train()
print(f"TRAIN_DONE {MODE} in {(time.time() - t0) / 60:.1f} min")

final_rep_path = os.path.join(OUT5, "final_report.json")
if os.path.exists(final_rep_path):
    final_rep = json.load(open(final_rep_path))
    print("\nFINAL_REPORT:")
    print(json.dumps(final_rep, indent=2))
    if final_rep["shares_ok"] and (final_rep["agree_sub3"] is None or final_rep["agree_sub3"] >= 0.95):
        print(f"\nSUBMISSION_READY -> {TOP_SUBMISSION_CSV}")


In [ ]:
# Cell 4 — Final Verification & Kaggle Submission Gate Check
import os, json, pandas as pd
sub_path = "/kaggle/working/submission.csv" if os.path.exists("/kaggle/working") else "/content/submission.csv"
rep_path = os.path.join(OUT5, "final_report.json")
assert os.path.exists(sub_path), f"Missing {sub_path}"
sub = pd.read_csv(sub_path, dtype=str)
assert list(sub.columns) == ["id", "label"] and len(sub) == 6000 and sub.id.is_unique
assert sub.id.tolist() == test.id.tolist() and sub.label.isin(LABELS).all()
print("Verified submission file:", sub_path, "| rows:", len(sub))
print(sub.label.value_counts(normalize=True).round(4))
if os.path.exists(rep_path):
    print("\nFinal Report:")
    print(open(rep_path).read())
